# Boreas / OpenPCDet — Kaggle T4 x2
Settings: GPU T4 x2, Internet ON. Add Input bản code hiện tại chứa các sửa đổi Boreas (không dùng upstream) và data đã prepare. Sửa CODE_INPUT, DATA_ROOT. Baseline Python 3.9–3.12, nvcc 12.x, torch2.5.1/cu121; chưa thực thi trên Kaggle. Chạy thử 1 epoch trước, sau đó đổi EPOCHS=80 và TAG. BATCH_PER_GPU=1 nghĩa là batch tổng=2; VRAM không gộp.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json
CODE_INPUT="/kaggle/input/your-code-dataset/openPCDet"
DATA_ROOT="/kaggle/input/your-boreas-dataset/boreas_openpcdet"
MODEL="pointpillar"
BEAMS=128
EPOCHS=1
BATCH_PER_GPU=1
WORKERS=2
SEED=42
TAG="kaggle_t4x2_smoke"
CKPT=""
RESUME=False
REPO=Path("/kaggle/working/openPCDet")
PY="/kaggle/working/pcdet-env/bin/python"
def run(args,cwd=None,env=None):
    print("Running:", " ".join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),cwd=cwd,env=env,check=True)
run(["nvidia-smi"])
assert not(CKPT and RESUME)
if not REPO.exists():
    assert (Path(CODE_INPUT)/"setup.py").is_file(),"Sửa CODE_INPUT"
    shutil.copytree(CODE_INPUT,REPO,ignore=shutil.ignore_patterns(".git","build","data","output","__pycache__","*.so","*.egg-info"))
assert (REPO/"tools/boreas_benchmark/beam_config.py").is_file(),"Cần bản code Boreas hiện tại"


In [ ]:
assert (3,9)<=sys.version_info[:2]<=(3,12),"Baseline cần Python 3.9–3.12"
nvcc=shutil.which("nvcc")
assert nvcc,"Cần CUDA toolkit/nvcc"
v=subprocess.check_output([nvcc,"--version"],text=True)
print(v)
assert "release 12." in v,"Baseline cần nvcc 12.x; đổi Kaggle image hoặc dependencies"
os.environ.update(CUDA_HOME=str(Path(nvcc).resolve().parents[1]),TORCH_CUDA_ARCH_LIST="7.5",MAX_JOBS="2")
if not Path(PY).exists(): run([sys.executable,"-m","venv","/kaggle/working/pcdet-env"])
run([PY,"-m","pip","install","pip","setuptools","wheel","ninja"])
run([PY,"-m","pip","install","torch==2.5.1","torchvision==0.20.1","--index-url","https://download.pytorch.org/whl/cu121"])
run([PY,"-m","pip","install","numpy==1.26.4","numba==0.60.0","spconv-cu120==2.3.8","tensorboardX","easydict","pyyaml","scikit-image","tqdm","SharedArray","opencv-python-headless==4.10.0.84","pyquaternion","pandas","scipy","av","kornia"])
run([PY,"setup.py","build_ext","--inplace"],cwd=REPO)
run([PY,"-m","pip","install","-e",".","--no-build-isolation","--no-deps"],cwd=REPO)


In [ ]:
assert BEAMS in (128,64,32,16)
target=Path(DATA_ROOT)/str(BEAMS)
for name in ["manifest.json","custom_infos_train.pkl","custom_infos_val.pkl","ImageSets/train.txt","ImageSets/val.txt","points","labels"]:
    assert (target/name).exists(),str(target/name)
m=json.loads((target/"manifest.json").read_text())
assert m.get("beam_count")==BEAMS and not m.get("smoke_test_only")
for split in ("train","val"):
    ids=(target/f"ImageSets/{split}.txt").read_text().split()
    assert ids and (target/"points"/f"{ids[0]}.npy").is_file()
    print(split,len(ids))
run([PY,"-c","import torch; import spconv.pytorch; from pcdet.models import build_network; from pcdet.ops.iou3d_nms import iou3d_nms_cuda; assert torch.cuda.device_count()==2; print(torch.__version__,torch.version.cuda); print([torch.cuda.get_device_name(i) for i in range(2)])"],cwd=REPO)


In [ ]:
check=REPO/"kaggle_ddp_check.py"
check.write_text("""import os,torch
import torch.distributed as d
from datetime import timedelta
r=int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(r)
d.init_process_group('nccl',timeout=timedelta(seconds=120))
x=torch.ones(1,device='cuda')
d.all_reduce(x)
assert x.item()==2
print('NCCL OK GPU',r,flush=True)
d.destroy_process_group()
""")
run([PY,"-m","torch.distributed.run","--standalone","--nnodes=1","--nproc_per_node=2",check],cwd=REPO)


In [ ]:
assert EPOCHS>=1 and BATCH_PER_GPU>=1
assert (REPO/f"tools/cfgs/boreas_models/{MODEL}.yaml").is_file()
if CKPT: assert Path(CKPT).is_file()
env=os.environ.copy()
env.update(CUDA_VISIBLE_DEVICES="0,1",OMP_NUM_THREADS="1",NCCL_DEBUG="WARN")
args=[PY,"-m","torch.distributed.run","--standalone","--nnodes=1","--nproc_per_node=2","train.py","--launcher","pytorch","--cfg_file",f"cfgs/boreas_models/{MODEL}.yaml","--batch_size",str(2*BATCH_PER_GPU),"--epochs",str(EPOCHS),"--workers",str(WORKERS),"--fix_random_seed","--seed",str(SEED),"--no_amp","--boreas_beams",str(BEAMS),"--extra_tag",TAG,"--ckpt_save_interval","1","--max_ckpt_save_num","3","--range_eval"]
if CKPT: args+=["--ckpt",CKPT]
elif RESUME: args+=["--resume"]
args+=["--set","DATA_CONFIG.DATA_PATH",str(target)]
run(args,cwd=REPO/"tools",env=env)


In [ ]:
for p in sorted((REPO/"output").rglob("*.pth")):
    print(p,round(p.stat().st_size/1024**2,1),"MiB")
for p in sorted((REPO/"output").rglob("accuracy.json")):
    print(p)
    print(p.read_text()[:3000])


Save Version với output và kiểm tra checkpoint trong /kaggle/working/openPCDet/output. Phiên sau Add Input checkpoint rồi đặt CKPT; EPOCHS là tổng epoch mục tiêu; giữ model/beam/seed/batch/precision. RESUME chỉ tìm output hiện tại. Benchmark mới: đổi TAG, CKPT rỗng. MODEL: pointpillar, pointpillar_lite, second, centerpoint, pv_rcnn, voxelnext. FP32 mặc định phù hợp PV-RCNN hiện tại. Nếu OOM batch1/GPU, cần model nhẹ hơn hoặc sửa representation có ghi nhận benchmark. Nguồn: https://docs.pytorch.org/docs/stable/elastic/run.html ; https://github.com/traveller59/spconv